In [33]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn import set_config
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import os

In [2]:
# read in and split data
parent_dir = os.path.dirname(os.getcwd())
data = pd.read_excel(os.path.join(parent_dir, 'Merged_HOA_ARC_ECU_WCP_allViolenceScores.xlsx'))

print(data.shape)
data_dna = data[data.columns[12:28]].dropna()
print(data_dna.shape)
x = data_dna.copy().drop('violence_score', axis=1)
print(x.shape)

y = data_dna['violence_score'].copy()
print(y.shape)
data_dna.info()

(1058, 33)
(902, 16)
(902, 15)
(902,)
<class 'pandas.DataFrame'>
RangeIndex: 902 entries, 156 to 1057
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   WeakGov          902 non-null    int64  
 1   FishPop          902 non-null    int64  
 2   EcoChngeOther    902 non-null    int64  
 3   GroundsLim       902 non-null    int64  
 4   ForeignFisher    902 non-null    int64  
 5   IllegalFishing   902 non-null    int64  
 6   IncrEfficiency   902 non-null    int64  
 7   IncrPressure     902 non-null    float64
 8   OpsScales        902 non-null    int64  
 9   Markets          902 non-null    int64  
 10  Poverty          902 non-null    int64  
 11  FoodInsecurity   902 non-null    int64  
 12  Marginalization  902 non-null    int64  
 13  StratLoc         902 non-null    int64  
 14  MaritimeCrime    902 non-null    int64  
 15  violence_score   902 non-null    int64  
dtypes: float64(1), int64(15)
memory 

In [5]:
train_data, test_data = train_test_split(data_dna, test_size=0.2, random_state=117)
# separate features and labels
X_train = train_data.drop(columns=['violence_score'])
y_train = train_data['violence_score']
X_test = test_data.drop(columns=['violence_score'])
y_test = test_data['violence_score']


In [66]:
# start testing neural network
params = {
    'hidden_layer_sizes': (50,),  # 1 hidden layer with 50 neurons
    'activation': 'relu',           # ReLU activation function
    'solver': 'adam',              # Adam optimizer
    'alpha': 0.05,                # L2 regularization
    'early_stopping': True,         # stop early to prevent overfitting
    'validation_fraction': 0.1,     # use 10% of training data for validation
    'n_iter_no_change': 10,         # stop if no improvement for 10 iterations
    'max_iter': 2000,               # maximum number of iterations
    'random_state': 117             # for reproducibility
}

nn_pipeline = ImbPipeline([('smote', SMOTE(random_state=117, sampling_strategy='auto')),
                           ('nn', MLPClassifier(**params))])

nn_pipeline.fit(X_train, y_train)
y_pred = nn_pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6298342541436464
              precision    recall  f1-score   support

           1       0.74      0.78      0.76        81
           2       0.67      0.53      0.59        77
           3       0.29      0.43      0.34        23

    accuracy                           0.63       181
   macro avg       0.57      0.58      0.57       181
weighted avg       0.65      0.63      0.64       181



In [49]:
#new base model with different parameters for tuning
base_params = {
    'random_state': 117,
    'early_stopping': False,
    'validation_fraction': 0.1,
    'n_iter_no_change': 10
}
base_nn_pipeline = ImbPipeline([('smote', SMOTE(random_state=117, sampling_strategy='auto')),
                                ('nn', MLPClassifier(**base_params))])
param_distributions = {
    'nn__hidden_layer_sizes': [(32,),(64,),(100,), (200,)],
    'nn__activation': ['relu', 'tanh'],
    'nn__solver': ['adam', 'lbfgs'],
    'nn__alpha': [0.01, 0.05, 0.1, 0.5],
    'nn__max_iter': [2000, 3000]
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=117)

In [50]:
# trying with grid search
nn_grid_search = GridSearchCV(estimator=base_nn_pipeline, param_grid=param_distributions,
                              cv=cv_strategy, scoring='f1_macro', n_jobs=-1, verbose=1)
nn_grid_search.fit(X_train, y_train)

print("Best Parameters from Grid Search:", nn_grid_search.best_params_)
best_nn_model_grid = nn_grid_search.best_estimator_
y_pred_best_grid = best_nn_model_grid.predict(X_test)
print("Best Model Accuracy from Grid Search:", accuracy_score(y_test, y_pred_best_grid))
print(classification_report(y_test, y_pred_best_grid))

Fitting 3 folds for each of 128 candidates, totalling 384 fits
Best Parameters from Grid Search: {'nn__activation': 'tanh', 'nn__alpha': 0.01, 'nn__hidden_layer_sizes': (200,), 'nn__max_iter': 2000, 'nn__solver': 'adam'}
Best Model Accuracy from Grid Search: 0.5524861878453039
              precision    recall  f1-score   support

           1       0.75      0.67      0.71        81
           2       0.57      0.45      0.51        77
           3       0.23      0.48      0.31        23

    accuracy                           0.55       181
   macro avg       0.52      0.53      0.51       181
weighted avg       0.61      0.55      0.57       181



In [ ]:
# try tuning NN with randomized search
nn_random_search = RandomizedSearchCV(estimator=base_nn_pipeline, param_distributions=param_distributions,
                                    n_iter=100, cv=cv_strategy, scoring='f1_macro', random_state=117,
                                    n_jobs=-1,verbose=1)

nn_random_search.fit(X_train, y_train)
print("Best Parameters:", nn_random_search.best_params_)
best_nn_model = nn_random_search.best_estimator_
y_pred_best = best_nn_model.predict(X_test)
print("Best Model Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))


x:\anaconda\envs\WWFnn\Lib\site-packages\sklearn\model_selection\_search.py:326: UserWarning: The total space of parameters 96 is smaller than n_iter=100. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters: {'nn__solver': 'lbfgs', 'nn__max_iter': 3000, 'nn__hidden_layer_sizes': (100,), 'nn__alpha': 0.1, 'nn__activation': 'tanh'}
Best Model Accuracy: 0.5580110497237569
              precision    recall  f1-score   support

           1       0.77      0.68      0.72        81
           2       0.59      0.45      0.51        77
           3       0.22      0.48      0.30        23

    accuracy                           0.56       181
   macro avg       0.53      0.54      0.51       181
weighted avg       0.63      0.56      0.58       181

